In [4]:
import requests

remote_pdf_url = "https://arxiv.org/pdf/1709.00666.pdf"
pdf_filename = "../data/ch02-downloaded.pdf"

response = requests.get(remote_pdf_url)

if response.status_code == 200:
    with open(pdf_filename, "wb") as pdf_file:
        pdf_file.write(response.content)
else:
    print("Failed to download the PDF. Status code:", response.status_code)

In [5]:
import pdfplumber

text = ""

with pdfplumber.open(pdf_filename) as pdf:
    for page in pdf.pages:
        text += page.extract_text()

print(text[0:20])

Einstein’s Patents a


In [12]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=40)
chunks = text_splitter.split_text(text)

chunks[0:2]

['Einstein’s Patents and Inventions\nAsis Kumar Chaudhuri\nVariable Energy Cyclotron Centre\n1‐AF Bidhan Nagar, Kolkata‐700 064\nAbstract: Times magazine selected Albert Einstein, the German born Jewish Scientist as the person of the 20th\ncentury. Undoubtedly, 20th century was the age of science and Einstein’s contributions in unravelling mysteries\nof nature was unparalleled. However, few are aware that Einstein was also a great inventor. He and his',
 'collaborators had patented a wide variety of inventions in several counties. After a brief description of Einstein’s\nlife, his collaborators, his inventions and patents will be discussed.\nFigure 1. Einstein in 1921 portrait after receiving Nobel Prize\n1. Introduction\nTowards the end of the last century, Times Magazine asked some of the World’s leading\npersonalities to pick their choice for the person of the century. The magazine compiled a list 100 most']

In [13]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = model.encode(chunks)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1822.24it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [14]:
print(embeddings[0][0:3])
print(len(embeddings))
print(len(embeddings[0]))

[-0.04625868  0.07410035 -0.05072907]
99
384


In [10]:
from neo4j import GraphDatabase
driver = GraphDatabase.driver("neo4j://localhost:7687", auth=("neo4j", "your_password"))

driver.verify_connectivity()

In [11]:
driver.execute_query("""CREATE VECTOR INDEX pdf IF NOT EXISTS
FOR (c:Chunk)
ON c.embedding""")

EagerResult(records=[], summary=<neo4j._work.summary.ResultSummary object at 0x324e05400>, keys=[])

In [15]:
# Add to neo4j
cypher_query = '''
WITH $chunks as chunks, range(0, size($chunks)) AS index
UNWIND index AS i
WITH i, chunks[i] AS chunk, $embeddings[i] AS embedding
MERGE (c:Chunk {index: i})
SET c.text = chunk, c.embedding = embedding
'''

driver.execute_query(cypher_query, chunks=chunks, embeddings=embeddings)

EagerResult(records=[], summary=<neo4j._work.summary.ResultSummary object at 0x324f35590>, keys=[])

In [16]:
records, _, _ = driver.execute_query("MATCH (c:Chunk) WHERE c.index = 0 RETURN c.embedding, c.text")

print(records[0]["c.text"][0:30])
print(records[0]["c.embedding"][0:3])

Einstein’s Patents and Inventi
[-0.046258676797151566, 0.07410034537315369, -0.050729066133499146]


In [18]:
question = "At what time was Einstein really interested in experimental works?"
question_embedding = model.encode(question)

In [19]:
query = '''
CALL db.index.vector.queryNodes('pdf', $k, $question_embedding) YIELD node AS hits, score
RETURN hits.text AS text, score, hits.index AS index
'''
similar_records, _, _ = driver.execute_query(query, question_embedding=question_embedding, k=4)

for record in similar_records:
    print(record["text"])
    print(record["score"], record["index"])
    print("======")

emphasized that his main occupation was theoretical physics. The inventions he worked upon were
his diversions. In his unproductive times he used to work upon on solving mathematical problems (not
related to his ongoing theoretical investigations) or took upon some practical problem. As shown in
Table. 2, Einstein was involved in three major inventions; (i) refrigeration system with Leo Szilard, (ii)
Sound reproduction system with Rudolf Goldschmidt and (iii) automatic camera with Gustav Bucky.
0.8022477626800537 49
Einstein’s Patents and Inventions
Asis Kumar Chaudhuri
Variable Energy Cyclotron Centre
1‐AF Bidhan Nagar, Kolkata‐700 064
Abstract: Times magazine selected Albert Einstein, the German born Jewish Scientist as the person of the 20th
century. Undoubtedly, 20th century was the age of science and Einstein’s contributions in unravelling mysteries
of nature was unparalleled. However, few are aware that Einstein was also a great inventor. He and his
0.7982113361358643 0
Being a m

In [20]:
from openai import OpenAI

open_ai_client = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama"
)

In [ ]:
system_message = "You're en Einstein expert, but can only use the provided documents to respond to the questions."
user_message = f"""
Use the following documents to answer the question that will follow:
{[doc["text"] for doc in similar_records]}

---

The question to answer using information only from the above documents: {question}
"""

print("Question:", question)

stream = open_ai_client.chat.completions.create(
    model="llama3.2",
    messages=[
        {"role": "system", "content": system_message},
        {"role": "user", "content": user_message}
    ],
    stream=True,
)
for chunk in stream:
    print(chunk.choices[0].delta.content or "", end="")

Question: At what time was Einstein really interested in experimental works?
According to the documents, Einstein was genuinely interested in experimental works during his ETH days. The exact time period is not explicitly mentioned, but it is implied to be from his ETH days, likely during his younger years.

In [24]:
stream = open_ai_client.chat.completions.create(
    model="llama3.2",
    messages=[
        {"role": "system", "content": system_message},
        {"role": "user", "content": user_message},
        {"role": "assistant", "content": "According to the documents, Einstein was genuinely interested in experimental works during his ETH days. The exact time period is not explicitly mentioned, but it is implied to be from his ETH days, likely during his younger years."},
        {"role": "user", "content": "What is ETH?"}
    ],
    stream=True,
)
for chunk in stream:
    print(chunk.choices[0].delta.content or "", end="")

The text does not explicitly state what ETH stands for, but based on the context and the mention of Einstein's ETH days, it is likely that ETH refers to the École Polytechnique Fédérale de Zurich, a Swiss university where Einstein studied.

In [25]:
try :
    driver.execute_query(f"CREATE FULLTEXT INDEX ftPdfChunk FOR (c:Chunk) ON EACH [c.text]")
except:
    print("Fulltext Index already exists")

In [26]:
hybrid_query = '''
CALL {
    // vector index
    CALL db.index.vector.queryNodes('pdf', $k, $question_embedding) YIELD node, score
    WITH collect({node:node, score:score}) AS nodes, max(score) AS max
    UNWIND nodes AS n
    // We use 0 as min
    RETURN n.node AS node, (n.score / max) AS score
    UNION
    // keyword index
    CALL db.index.fulltext.queryNodes('ftPdfChunk', $question, {limit: $k})
    YIELD node, score
    WITH collect({node:node, score:score}) AS nodes, max(score) AS max
    UNWIND nodes AS n
    // We use 0 as min
    RETURN n.node AS node, (n.score / max) AS score
}
// deduplicate results from both queries and return top k
WITH node, max(score) AS score ORDER BY score DESC LIMIT $k
RETURN node, score
'''
similar_hybrid_records, _, _ = driver.execute_query(hybrid_query, question_embedding=question_embedding, question=question, k=4)

for record in similar_hybrid_records:
    print(record["node"]["text"])
    print(record["score"], record["node"]["index"])
    print("======")

Received notification from DBMS server: {severity: WARNING} {code: Neo.ClientNotification.Statement.FeatureDeprecationWarning} {category: DEPRECATION} {title: This feature is deprecated and will be removed in future versions.} {description: CALL subquery without a variable scope clause is deprecated. Use CALL () { ... }} {position: line: 2, column: 1, offset: 1} for query: "\nCALL {\n    // vector index\n    CALL db.index.vector.queryNodes('pdf', $k, $question_embedding) YIELD node, score\n    WITH collect({node:node, score:score}) AS nodes, max(score) AS max\n    UNWIND nodes AS n\n    // We use 0 as min\n    RETURN n.node AS node, (n.score / max) AS score\n    UNION\n    // keyword index\n    CALL db.index.fulltext.queryNodes('ftPdfChunk', $question, {limit: $k})\n    YIELD node, score\n    WITH collect({node:node, score:score}) AS nodes, max(score) AS max\n    UNWIND nodes AS n\n    // We use 0 as min\n    RETURN n.node AS node, (n.score / max) AS score\n}\n// deduplicate results fr

emphasized that his main occupation was theoretical physics. The inventions he worked upon were
his diversions. In his unproductive times he used to work upon on solving mathematical problems (not
related to his ongoing theoretical investigations) or took upon some practical problem. As shown in
Table. 2, Einstein was involved in three major inventions; (i) refrigeration system with Leo Szilard, (ii)
Sound reproduction system with Rudolf Goldschmidt and (iii) automatic camera with Gustav Bucky.
1.0 49
Being a manufacturer’s son, Einstein grew upon in an environment of machines and instruments.
When his father’s company obtained the contract to illuminate Munich city during beer festival, he
was actively engaged in execution of the contract. In his ETH days Einstein was genuinely interested
in experimental works. He wrote to his friend, “most of the time I worked in the physical laboratory,
fascinated by the direct contact with observation.” Einstein's Uncle Jacob was enterprising and
1